# 0. Install necessary packages.
## RESTART AFTER THIS CELL IS RUN.

In [1]:
!pip install tensorflow
!pip install -U "jax[tpu]"
!pip install -U flax
!pip install -U jaxlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 155.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 171.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 185.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.2/225.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.4/198.4 MB 18.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 MB 39.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 160.8 MB/s eta 0:00:00
  Attempting uninstall: libtpu
    Found existing installation: libtpu 0.0.21
    Uninstalling libtpu-0.0.21:
      Successfully uninstalled libtpu-0.0.21
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.7.2
    Unin

### Be sure to restart the noteboook after the above cell is run

# 1. Set up Google Colab and Link Google Drive

In [ ]:
# Set up Google Colab and link Google Drive
import os
import signal
from google.colab import drive

# 1. Mount the full drive
drive.mount('/content/drive')

# 2. Change directory to your project folder
print('Current working directory:')
os.makedirs('/content/drive/My Drive/demo_drive_folder/saved_models', exist_ok = True)
os.chdir('/content/drive/My Drive/demo_drive_folder/')

# 3. git clone bonsai
!git clone https://github.com/jax-ml/bonsai.git

# Verify it worked
print(os.getcwd())
print(os.listdir())

# Load jax
import jax.tools.colab_tpu
import jax
from jax.extend import backend
print("JAX platform:", backend.get_backend().platform)
print("JAX devices:", jax.devices())

Mounted at /content/drive
Current working directory:
Cloning into 'bonsai'...
remote: Enumerating objects: 2172, done.
remote: Counting objects: 100% (828/828), done.
remote: Compressing objects: 100% (433/433), done.
remote: Total 2172 (delta 621), reused 396 (delta 395), pack-reused 1344 (from 2)
Receiving objects: 100% (2172/2172), 8.71 MiB | 29.53 MiB/s, done.
Resolving deltas: 100% (1246/1246), done.
/content/drive/My Drive/demo_drive_folder
['saved_models', 'bonsai']


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX platform: tpu
JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


In [2]:
import tensorflow_datasets as tfds  # TFDS to download EuroSAT.
import tensorflow as tf  # TensorFlow / `tf.data` operations.

tf.random.set_seed(0)  # Set the random seed for reproducibility.

train_steps = 2000
eval_every = 400
batch_size = 32

# EuroSAT/RGB: 27,000 Sentinel-2 satellite images, 64x64, 10 land-use classes.
# Classes: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial,
#          Pasture, PermanentCrop, Residential, River, SeaLake
train_ds: tf.data.Dataset = tfds.load('eurosat/rgb', split='train[:80%]')
test_ds: tf.data.Dataset  = tfds.load('eurosat/rgb', split='train[80%:]')

EUROSAT_CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVeg', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

def preprocess(sample):
    image = tf.cast(sample['image'], tf.float32) / 255.0  # Normalize to [0, 1].
    return {'image': image, 'label': sample['label']}

train_ds = train_ds.map(preprocess)
test_ds  = test_ds.map(preprocess)

train_ds = train_ds.repeat().shuffle(1024)
train_ds = train_ds.batch(batch_size, drop_remainder=True).take(train_steps).prefetch(1)
test_ds  = test_ds.batch(batch_size, drop_remainder=True).prefetch(1)


# 2. Load EuroSAT — Sentinel-2 Land Use Classification

**EuroSAT** is a benchmark dataset built from ESA Sentinel-2 satellite imagery.
Each image is 64×64 pixels with 3 RGB channels, covering 10 land-use / land-cover classes
across Europe. The task of distinguishing crop types, forests, rivers, and urban areas from
satellite imagery is directly relevant to environmental monitoring, agricultural assessment,
and climate impact studies.

| Class | Examples |
|---|---|
| AnnualCrop | 3000 |
| Forest | 3000 |
| HerbaceousVegetation | 3000 |
| Highway | 2500 |
| Industrial | 2500 |
| Pasture | 2000 |
| PermanentCrop | 2500 |
| Residential | 3000 |
| River | 2500 |
| SeaLake | 3000 |


In [3]:
from flax import nnx  # The Flax NNX API.
from functools import partial
from typing import Optional

class CNN(nnx.Module):
  """A simple CNN for EuroSAT (64x64 RGB satellite images, 10 classes)."""

  def __init__(self, *, rngs: nnx.Rngs):
    self.conv1 = nnx.Conv(3, 32, kernel_size=(3, 3), rngs=rngs)  # 3-channel RGB input
    self.batch_norm1 = nnx.BatchNorm(32, rngs=rngs)
    self.dropout1 = nnx.Dropout(rate=0.025)
    self.conv2 = nnx.Conv(32, 64, kernel_size=(3, 3), rngs=rngs)
    self.batch_norm2 = nnx.BatchNorm(64, rngs=rngs)
    self.avg_pool = partial(nnx.avg_pool, window_shape=(2, 2), strides=(2, 2))
    # After two (conv SAME + 2x2 pool) blocks on 64x64: 64→32→16, so 16*16*64=16384
    self.linear1 = nnx.Linear(16384, 256, rngs=rngs)
    self.dropout2 = nnx.Dropout(rate=0.025)
    self.linear2 = nnx.Linear(256, 10, rngs=rngs)

  def __call__(self, x, rngs: Optional[nnx.Rngs] = None):
    x = self.avg_pool(nnx.relu(self.batch_norm1(self.dropout1(self.conv1(x), rngs=rngs))))
    x = self.avg_pool(nnx.relu(self.batch_norm2(self.conv2(x))))
    x = x.reshape(x.shape[0], -1)  # flatten
    x = nnx.relu(self.dropout2(self.linear1(x), rngs=rngs))
    x = self.linear2(x)
    return x

# Instantiate the model.
model = CNN(rngs=nnx.Rngs(0))
# Visualize it.
nnx.display(model)


# Run the model

In [4]:
import jax.numpy as jnp  # JAX NumPy

y = model(jnp.ones((1, 64, 64, 3)), nnx.Rngs(0))  # 64x64 RGB input
y


# 4. Create the optimizer and define some metrics

In [5]:
import optax

learning_rate = 0.005
momentum = 0.9

optimizer = nnx.Optimizer(
  model, optax.adamw(learning_rate, momentum), wrt=nnx.Param
)
metrics = nnx.MultiMetric(
  accuracy=nnx.metrics.Accuracy(),
  loss=nnx.metrics.Average('loss'),
)

nnx.display(optimizer)

# 5. Define training step functions

In [6]:
def loss_fn(model: CNN, rngs: nnx.Rngs, batch):
  logits = model(batch['image'], rngs)
  loss = optax.softmax_cross_entropy_with_integer_labels(
    logits=logits, labels=batch['label']
  ).mean()
  return loss, logits

@nnx.jit
def train_step(model: CNN, optimizer: nnx.Optimizer, metrics: nnx.MultiMetric, rngs: nnx.Rngs, batch):
  """Train for a single step."""
  grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
  (loss, logits), grads = grad_fn(model, rngs, batch)
  metrics.update(loss=loss, logits=logits, labels=batch['label'])  # In-place updates.
  optimizer.update(model, grads)  # In-place updates.

@nnx.jit
def eval_step(model: CNN, metrics: nnx.MultiMetric, rngs: nnx.Rngs, batch):
  loss, logits = loss_fn(model, rngs, batch)
  metrics.update(loss=loss, logits=logits, labels=batch['label'])  # In-place updates.

# 6. Train and evaluate the model

In [7]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

metrics_history = {
  'train_loss': [],
  'train_accuracy': [],
  'test_loss': [],
  'test_accuracy': [],
}

rngs = nnx.Rngs(0)

for step, batch in enumerate(train_ds.as_numpy_iterator()):
  # Run the optimization for one step and make a stateful update to the following:
  # - The train state's model parameters
  # - The optimizer state
  # - The training loss and accuracy batch metrics
  model.train() # Switch to train mode
  train_step(model, optimizer, metrics, rngs, batch)

  if step > 0 and (step % eval_every == 0 or step == train_steps - 1):  # One training epoch has passed.
    # Log the training metrics.
    for metric, value in metrics.compute().items():  # Compute the metrics.
      metrics_history[f'train_{metric}'].append(value)  # Record the metrics.
    metrics.reset()  # Reset the metrics for the test set.

    # Compute the metrics on the test set after each training epoch.
    model.eval() # Switch to eval mode
    for test_batch in test_ds.as_numpy_iterator():
      eval_step(model, metrics, rngs, test_batch)

    # Log the test metrics.
    for metric, value in metrics.compute().items():
      metrics_history[f'test_{metric}'].append(value)
    metrics.reset()  # Reset the metrics for the next training epoch.

    clear_output(wait=True)
    # Plot loss and accuracy in subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.set_title('Loss')
    ax2.set_title('Accuracy')
    for dataset in ('train', 'test'):
      ax1.plot(metrics_history[f'{dataset}_loss'], label=f'{dataset}_loss')
      ax2.plot(metrics_history[f'{dataset}_accuracy'], label=f'{dataset}_accuracy')
    ax1.legend()
    ax2.legend()
    plt.show()

# 7. Perform inference on the test set

In [8]:
model.eval() # Switch to evaluation mode.

@nnx.jit
def pred_step(model: CNN, batch):
  logits = model(batch['image'], None)
  return logits.argmax(axis=1)

In [9]:
test_batch = test_ds.as_numpy_iterator().next()
pred = pred_step(model, test_batch)

fig, axs = plt.subplots(5, 5, figsize=(12, 12))
for i, ax in enumerate(axs.flatten()):
    ax.imshow(test_batch['image'][i])  # RGB: no cmap needed
    true_label = EUROSAT_CLASSES[test_batch['label'][i]]
    pred_label = EUROSAT_CLASSES[pred[i]]
    color = 'green' if pred[i] == test_batch['label'][i] else 'red'
    ax.set_title(f'pred: {pred_label}\ntrue: {true_label}', fontsize=7, color=color)
    ax.axis('off')
plt.tight_layout()


# 8. Export the model

In [10]:
from orbax.export import JaxModule, ExportManager, ServingConfig

In [11]:
def exported_predict(model, y):
    return model(y, None)

jax_module = JaxModule(model, exported_predict)

In [12]:
sig = [tf.TensorSpec(shape=(1, 64, 64, 3), dtype=tf.float32)]  # 64x64 RGB


In [13]:
export_mgr = ExportManager(jax_module, [
    ServingConfig('eurosat_server', input_signature=sig)
])

output_dir='saved_models/demo_model'
export_mgr.save(output_dir)


In [14]:
!ls saved_models/demo_model

# 9. Unmount Google Drive

In [15]:
drive.flush_and_unmount()
print('Safe to unmount from Google Drive.')

Safe to unmount from Google Drive.


# 10. Release TPU

In [ ]:
# This forces the kernel to exit, which triggers 
# the TPU driver to release the hardware lock immediately.
os.kill(os.getpid(), signal.SIGTERM)

: 

: 

: 